# Riemann Evidence Atlas research model replay

On Colab, set `ATLAS_COMMIT` to the full SHA of the public Atlas commit to inspect, then run the cells. Turn on `RUN_LEAN` for the frozen Lean/Mathlib replay. The numerical cells use Python's standard library. The last cell can download the pinned Lean binary and Mathlib cache. Reports are written under `outputs/research-model-replay`. GitHub Actions executes these same source cells with its own checkout. A GitHub notebook-cell run is recorded as a hosted runner check.


In [ ]:
from pathlib import Path
import os
import re
import subprocess
import sys

ATLAS_COMMIT = "" # @param {type:"string"}
repo = "https://github.com/Flamehaven-Labs/riemann-evidence-atlas.git"
if "ATLAS_REPLAY_ROOT" in os.environ:
    root = Path(os.environ["ATLAS_REPLAY_ROOT"]).resolve()
    expected = os.environ["ATLAS_REPLAY_COMMIT"]
else:
    if re.fullmatch(r"[0-9a-f]{40}", ATLAS_COMMIT) is None:
        raise ValueError("Enter a full Atlas Git commit SHA in ATLAS_COMMIT")
    expected = ATLAS_COMMIT
    root = Path("/content") / f"riemann-evidence-atlas-{expected[:12]}"
    if not root.exists():
        subprocess.run(["git", "init", str(root)], check=True)
        subprocess.run(["git", "-C", str(root), "fetch", "--depth=1", "--no-tags", repo, expected], check=True)
        subprocess.run(["git", "-C", str(root), "checkout", "--detach", "FETCH_HEAD"], check=True)

actual = subprocess.check_output(["git", "-C", str(root), "rev-parse", "HEAD"], text=True).strip()
if actual != expected:
    raise ValueError(f"Atlas checkout {actual} differs from requested commit {expected}")
if subprocess.check_output(["git", "-C", str(root), "status", "--porcelain", "--untracked-files=no"]):
    raise ValueError("Atlas replay requires a clean checkout of the requested commit")
output = Path(os.environ.get("ATLAS_REPLAY_OUTPUT", root / "outputs/research-model-replay")).resolve()
output.mkdir(parents=True, exist_ok=True)
(output / "source-commit.txt").write_text(actual + "\n", encoding="ascii")
print("Atlas commit", actual)


In [ ]:
subprocess.run([sys.executable, "-E", "-S", "tools/verify_research_models.py",
                "--output", str(output / "numerical-models.json")], cwd=root, check=True)


In [ ]:
RUN_LEAN = False # @param {type:"boolean"}
if RUN_LEAN:
    from hashlib import file_digest
    import shutil
    from urllib.request import urlretrieve

    target = root / "outputs" / "lean-4.33.1-linux"
    lean = target / "bin" / "lean"
    if not lean.is_file():
        archive = target.parent / "lean-4.33.1-linux.tar.zst"
        archive.parent.mkdir(parents=True, exist_ok=True)
        urlretrieve("https://github.com/leanprover/lean4/releases/download/v4.33.1/lean-4.33.1-linux.tar.zst", archive)
        with archive.open("rb") as stream:
            digest = file_digest(stream, "sha256").hexdigest()
        if digest != "890afd185370f85666025b883914ab4f4b339136f8c96167b69cfb62aecaf235":
            raise ValueError("Lean download SHA-256 mismatch")
        if shutil.which("zstd") is None:
            subprocess.run(["apt-get", "update", "-qq"], check=True)
            subprocess.run(["apt-get", "install", "-y", "-qq", "zstd"], check=True)
        subprocess.run(["tar", "--zstd", "-xf", str(archive), "-C", str(target.parent)], check=True)

    env = {**os.environ, "PATH": str(lean.parent) + os.pathsep + os.environ["PATH"]}
    project = root / "formal" / "landau"
    imports = subprocess.check_output([sys.executable, "-E", "-S", "tools/verify_model_proofs.py",
                                       "--mathlib-imports"], cwd=root, text=True).splitlines()
    subprocess.run([str(lean.with_name("lake")), "exe", "cache", "get", *imports],
                   cwd=project, env=env, check=True)
    subprocess.run([sys.executable, "-E", "-S", "tools/verify_model_proofs.py", "--lean", str(lean),
                    "--mathlib-project", str(project), "--output-dir", str(output / "lean")],
                   cwd=root, env=env, check=True)
else:
    print("Lean replay disabled in this notebook run")
